# Mashi–French data extraction

Adapted for the AfriCompLing Summer School 2026.

This notebook creates Mashi–French candidate bitext from three sources:

1. verse-aligned Exodus chapters from the Mashi and French eBible editions;
2. Mashi–French words and example sentences returned by selected searches on murhula.com;
3. Mashi–French rows in `deuteronome-dictionnaire-contextuel-mashi-hebreu-francais.pdf` (the Hebrew column is intentionally ignored).

The notebook targets at most 500 Bible pairs. Smaller results are valid. Web and PDF candidates are marked for human review.

## 1. Install and import packages

In Colab, upload the supplied PDF to `/content/` before running the PDF section.

In [ ]:
!pip -q install beautifulsoup4 pandas pypdf requests tqdm

In [ ]:
import json
import re
import time
import unicodedata
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
import requests
from bs4 import BeautifulSoup, NavigableString, Tag
from pypdf import PdfReader
from tqdm.auto import tqdm

HEADERS = {
    "User-Agent": "AfriCompLing-Mashi-research/1.0 (small academic request set)"
}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def clean_text(value):
    """Normalize Unicode and whitespace without removing Mashi diacritics."""
    value = unicodedata.normalize("NFC", str(value or ""))
    # Some PDF emoji are exposed as isolated UTF-16 surrogates by pypdf.
    value = value.encode("utf-8", "replace").decode("utf-8")
    value = value.replace("\u00a0", " ")
    return re.sub(r"\s+", " ", value).strip(" \t\r\n•-")

def valid_pair(mashi, french):
    mashi, french = clean_text(mashi), clean_text(french)
    return (
        len(mashi) >= 2
        and len(french) >= 2
        and bool(re.search(r"[A-Za-zÀ-ÖØ-öø-ÿ]", mashi))
        and bool(re.search(r"[A-Za-zÀ-ÖØ-öø-ÿ]", french))
        and mashi.casefold() != french.casefold()
    )

def deduplicate(df):
    if df.empty:
        return df
    out = df.copy()
    if "quality_flags" not in out.columns:
        out["quality_flags"] = ""
    out["mashi"] = out["mashi"].map(clean_text)
    out["french"] = out["french"].map(clean_text)
    out = out[out.apply(lambda row: valid_pair(row.mashi, row.french), axis=1)]
    return out.drop_duplicates(subset=["mashi", "french"]).reset_index(drop=True)

## 2. Verse-aligned Mashi–French Bible bitext

The Mashi Exodus pages use `span.verse` elements with IDs such as `V1`. The French Louis Segond pages use the same chapter/verse organization. We extract each edition independently and inner-join on `(book, chapter, verse)`—never on sentence order.

- Mashi source: `https://ebible.org/shr/EXO01.htm`
- French alignment source: `https://ebible.org/fraLSG/EXO01.htm`
- Mashi edition: CC BY 4.0 on its eBible details page.
- French Louis Segond 1910: public-domain edition on eBible.

The translations are not expected to have identical punctuation or sentence segmentation, so the unit is a **verse**, not necessarily one sentence.

In [ ]:
EBIBLE_NONVERSE_CLASSES = {
    "notemark", "s", "s1", "s2", "r", "d", "sp",
    "mt", "mt1", "ms", "ms1", "mr", "cl", "rem",
}

def text_until_next_verse(verse_span):
    """Collect text in document order, including poetry split across blocks."""
    parts = []
    for node in verse_span.next_elements:
        if isinstance(node, Tag) and (
            node.name in {"footer", "script"}
            or "tnav" in node.get("class", [])
            or "footnotes" in node.get("class", [])
        ):
            break
        if (
            isinstance(node, Tag)
            and node.name == "span"
            and "verse" in node.get("class", [])
        ):
            break
        if not isinstance(node, NavigableString):
            continue
        if verse_span in node.parents:
            continue
        parent_classes = {
            cls
            for parent in node.parents
            if isinstance(parent, Tag)
            for cls in parent.get("class", [])
        }
        if parent_classes & EBIBLE_NONVERSE_CLASSES:
            continue
        parts.append(str(node))
    return clean_text(" ".join(parts))

def extract_ebible_chapter(url, book, chapter, language):
    response = SESSION.get(url, timeout=30)
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    rows = []
    for span in soup.select('span.verse[id^="V"]'):
        match = re.fullmatch(r"V(\d+)", span.get("id", ""))
        if not match:
            continue
        text = text_until_next_verse(span)
        if text:
            rows.append({
                "book": book,
                "chapter": chapter,
                "verse": int(match.group(1)),
                language: text,
                f"{language}_url": url,
            })
    return rows

def build_exodus_bitext(max_pairs=500, pause_seconds=0.20):
    mashi_rows, french_rows = [], []
    for chapter in tqdm(range(1, 41), desc="Exodus chapters"):
        suffix = f"EXO{chapter:02d}.htm"
        mashi_url = f"https://ebible.org/shr/{suffix}"
        french_url = f"https://ebible.org/fraLSG/{suffix}"
        mashi_rows.extend(extract_ebible_chapter(mashi_url, "EXO", chapter, "mashi"))
        french_rows.extend(extract_ebible_chapter(french_url, "EXO", chapter, "french"))
        time.sleep(pause_seconds)

    mashi_df = pd.DataFrame(mashi_rows)
    french_df = pd.DataFrame(french_rows)
    aligned = mashi_df.merge(french_df, on=["book", "chapter", "verse"], how="inner")
    aligned["ref"] = aligned.apply(
        lambda row: f"{row.book} {row.chapter}:{row.verse}", axis=1
    )
    aligned["source"] = "eBible Exodus: shr + fraLSG"
    aligned["source_url"] = aligned["mashi_url"]
    aligned["translation_url"] = aligned["french_url"]
    aligned["unit_type"] = "verse"
    aligned["review_status"] = "needs_review"
    columns = [
        "source", "source_url", "translation_url", "ref",
        "mashi", "french", "unit_type", "review_status"
    ]
    return deduplicate(aligned[columns]).head(max_pairs)

bible_bitext = build_exodus_bitext(max_pairs=500)
print(f"Aligned Bible rows: {len(bible_bitext)}")
display(bible_bitext.head(10))

In [ ]:
# Alignment integrity checks
assert bible_bitext["ref"].is_unique
assert bible_bitext[["mashi", "french"]].notna().all().all()
assert len(bible_bitext) <= 500
print(bible_bitext["ref"].head().tolist())
print(bible_bitext[["mashi", "french"]].sample(min(5, len(bible_bitext)), random_state=42).to_string(index=False))

## 3. Mashi–French words and example sentences from murhula.com

murhula.com is an interactive dictionary, not a static downloadable dataset. This notebook therefore sends only a small, explicit list of searches, waits between requests, and saves provenance for each result. Expand the query list slowly and only with the site's permission for larger collection.

The parser handles:

- `tr_fnc` tables containing explicit Mashi and French lists;
- `tr_blc` tables containing Mashi/French word fields;
- example rows where the French example is followed by its Mashi example.

All murhula.com rows remain `needs_review` because dynamically generated verb analyses and examples may be contextual rather than exact translations.

In [ ]:
MURHULA_QUERIES = [
    "bonjour", "merci", "homme", "femme", "enfant", "père", "mère",
    "maison", "eau", "soleil", "jour", "nuit", "manger", "boire",
    "aimer", "venir", "partir", "travail", "main", "tête",
]

def cell_text(cell):
    return clean_text(cell.get_text(" ", strip=True))

def list_or_text(cell):
    items = [clean_text(li.get_text(" ", strip=True)) for li in cell.find_all("li")]
    items = [item for item in items if item]
    return items or [cell_text(cell)]

def direct_entry_title(table):
    parent = table.parent
    if not parent:
        return ""
    for span in parent.find_all("span", recursive=False):
        text = clean_text(span.get_text(" ", strip=True))
        if text and not text.lower().startswith(("dom.", "nat.")):
            return text
    return ""

def parse_murhula_table(table, query, url):
    records = []
    parsed_rows = []
    for tr in table.find_all("tr"):
        cells = tr.find_all("td", recursive=False)
        if len(cells) >= 2:
            parsed_rows.append((cell_text(cells[0]).casefold(), cells[1]))

    # Explicit paired Mashi and French lists.
    mashi_cells = [cell for label, cell in parsed_rows if label.rstrip(".") == "mashi"]
    french_cells = [cell for label, cell in parsed_rows if label.rstrip(".") in {"français", "francais"}]
    for m_cell, f_cell in zip(mashi_cells, french_cells):
        for mashi, french in zip(list_or_text(m_cell), list_or_text(f_cell)):
            if valid_pair(mashi, french):
                records.append((mashi, french, "word_or_phrase"))

    # Dictionary headword rows, including verbs.
    mashi_values, french_values = [], []
    for label, cell in parsed_rows:
        if label in {"verbe mashi", "mashi"}:
            mashi_values.extend(list_or_text(cell))
        elif label in {"verbe francais", "verbe français", "français", "francais"}:
            french_values.extend(list_or_text(cell))
    title = direct_entry_title(table)
    if mashi_values and not french_values and title:
        french_values = [title]
    for mashi in mashi_values:
        for french in french_values:
            if valid_pair(mashi, french):
                records.append((mashi, french, "lexicon"))

    # French example followed by an unlabeled or '/' Mashi example row.
    for idx, (label, cell) in enumerate(parsed_rows[:-1]):
        if label == "exemple":
            next_label, next_cell = parsed_rows[idx + 1]
            if next_label in {"", "/"}:
                french = cell_text(cell)
                mashi = re.sub(r"^(Exemple\s*:|\([^)]*\))\s*", "", cell_text(next_cell), flags=re.I)
                if valid_pair(mashi, french):
                    records.append((mashi, french, "example_sentence"))

    return [{
        "source": "murhula.com Mashi–French dictionary",
        "source_url": url,
        "translation_url": url,
        "ref": f"query={query}",
        "mashi": mashi,
        "french": french,
        "unit_type": unit_type,
        "review_status": "needs_review",
    } for mashi, french, unit_type in records]

def extract_murhula_queries(queries, pause_seconds=1.0):
    records = []
    for query in tqdm(queries, desc="murhula.com searches"):
        url = "https://murhula.com/index.php?" + urlencode({"neno": query})
        response = SESSION.get(url, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        for table in soup.select("table.tr_fnc, table.tr_blc"):
            records.extend(parse_murhula_table(table, query, url))
        time.sleep(pause_seconds)
    return deduplicate(pd.DataFrame(records))

murhula_bitext = extract_murhula_queries(MURHULA_QUERIES)
print(f"murhula.com candidate rows: {len(murhula_bitext)}")
display(murhula_bitext.head(20))

## 4. Extract Mashi and French from the trilingual PDF

The PDF has repeating layout tables with columns `Mashi | Hébreu (BHS) | Français | Réf.`. `pypdf` layout mode preserves these approximate columns. The extractor detects each header, slices subsequent rows at the detected column boundaries, discards the Hebrew column, and joins wrapped Mashi/French cells.

This document contains contextual phrases rather than a verse-complete Bible. Keep its output separate from the eBible verse corpus and manually review it. The PDF's own reuse licence is not stated inside the extraction workflow.

In [ ]:
PDF_NAME = "deuteronome-dictionnaire-contextuel-mashi-hebreu-francais.pdf"
PDF_PATH = Path("language_resources/mashi-shr/sources") / PDF_NAME
if not PDF_PATH.exists():
    # Colab fallback: upload the PDF to the notebook working directory.
    PDF_PATH = Path(PDF_NAME)
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {PDF_NAME}; keep it in the repository source folder or upload it to Colab."
    )

HEBREW_RE = re.compile(r"[\u0590-\u05FF\uFB1D-\uFB4F]")
REF_RE = re.compile(r"\b\d{1,2}:\d{1,2}(?:\s*[–-]\s*\d{1,2})?\b")

def strip_hebrew(text):
    text = re.sub(r"[\u0590-\u05FF\uFB1D-\uFB4F]+", " ", text)
    return clean_text(text)

def clean_pdf_cell(text):
    """Remove predictable page-layout debris without changing Mashi letters."""
    text = strip_hebrew(text)
    text = re.sub(r"^\s*\d+(?:\s*[–-]\s*\d+)?[.)]\s*", "", text)
    text = re.sub(r"(?<!\w)Dt(?!\w)", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return clean_text(text)

def pdf_quality_flags(mashi, french, ref):
    """Return conservative flags for rows that should be quarantined."""
    flags = []
    m_words, f_words = max(1, len(mashi.split())), max(1, len(french.split()))
    ratio = max(m_words / f_words, f_words / m_words)
    if not REF_RE.fullmatch(ref):
        flags.append("missing_reference")
    if ratio > 4:
        flags.append("extreme_length_ratio")
    if min(m_words, f_words) == 1 and max(m_words, f_words) >= 8:
        flags.append("very_short_side")
    if re.search(r"Voici la suite|rset\b|tableau trilingue", mashi + " " + french, re.I):
        flags.append("page_note")
    if len(re.findall(r"(?:^|\s)\d+(?:[.)]|\s*[–-]\s*\d+)", mashi)) > 1:
        flags.append("merged_rows")
    return flags

def find_table_columns(header):
    mashi = header.find("Mashi")
    hebrew = header.find("Hébreu")
    french = header.find("Français")
    ref = header.find("Réf")
    if min(mashi, hebrew, french, ref) < 0:
        return None
    # Header words are centered; these boundaries approximate the actual cells.
    hebrew_boundary = max(1, hebrew - 5)
    french_boundary = (hebrew + french) // 2
    ref_boundary = max(french_boundary + 1, ref - 4)
    return hebrew_boundary, french_boundary, ref_boundary

def extract_pdf_rows(pdf_path):
    reader = PdfReader(str(pdf_path))
    records = []
    for page_number, page in enumerate(tqdm(reader.pages, desc="PDF pages"), start=1):
        text = page.extract_text(extraction_mode="layout") or ""
        lines = text.splitlines()
        columns = None
        current = None

        def flush():
            nonlocal current
            if current:
                mashi = clean_pdf_cell(" ".join(current["mashi"]))
                french = clean_pdf_cell(" ".join(current["french"]))
                ref = clean_text(" ".join(current["ref"]))
                ref_match = REF_RE.search(ref)
                if valid_pair(mashi, french):
                    normalized_ref = ref_match.group(0) if ref_match else f"PDF page {page_number}"
                    flags = pdf_quality_flags(mashi, french, normalized_ref)
                    records.append({
                        "source": "Deutéronome dictionnaire contextuel Mashi–Hébreu–Français PDF",
                        "source_url": PDF_PATH.name,
                        "translation_url": PDF_PATH.name,
                        "ref": normalized_ref,
                        "mashi": mashi,
                        "french": french,
                        "unit_type": "context_phrase",
                        "review_status": "quarantine" if flags else "needs_review",
                        "quality_flags": ";".join(flags),
                    })
            current = None

        for line in lines:
            detected = find_table_columns(line)
            if detected:
                flush()
                columns = detected
                continue
            if columns is None:
                continue
            h_col, f_col, r_col = columns
            padded = line.ljust(r_col + 30)

            # Headings/notes terminate the active table.
            stripped_line = line.strip()
            if (
                stripped_line.startswith(("?", "•", "", "Merci "))
                or "Notes thématiques" in stripped_line
                or re.search(r"(?:^|\s)[A-F]\.\s", stripped_line)
            ):
                flush()
                columns = None
                continue

            # Hebrew characters give more reliable boundaries than visual columns.
            hebrew_positions = [m.start() for m in HEBREW_RE.finditer(line)]
            if hebrew_positions:
                first_hebrew, last_hebrew = hebrew_positions[0], hebrew_positions[-1]
                mashi = clean_text(line[:first_hebrew])
                tail = line[last_hebrew + 1:]
                ref_matches = list(REF_RE.finditer(tail))
                if ref_matches:
                    last_ref = ref_matches[-1]
                    french = clean_text(tail[:last_ref.start()])
                    ref = clean_text(tail[last_ref.start():])
                else:
                    french, ref = clean_text(tail), ""
            else:
                mashi = clean_text(padded[:h_col])
                french = clean_text(padded[f_col:r_col])
                ref = clean_text(padded[r_col:])

            # A new data row normally begins with Mashi plus Hebrew content.
            if mashi and hebrew_positions:
                first_letter = next((char for char in mashi if char.isalpha()), "")
                is_wrapped_row = bool(current and first_letter and first_letter.islower())
                if is_wrapped_row:
                    current["mashi"].append(mashi)
                    if french:
                        current["french"].append(french)
                    if ref:
                        current["ref"].append(ref)
                else:
                    flush()
                    current = {"mashi": [mashi], "french": [french], "ref": [ref]}
            elif current and hebrew_positions:
                # A long Hebrew cell may wrap; keep any French continuation.
                if french:
                    current["french"].append(french)
                if ref:
                    current["ref"].append(ref)
            elif current and not hebrew_positions:
                # Wrapped cells stay in their respective visual columns.
                if mashi:
                    current["mashi"].append(mashi)
                if french:
                    current["french"].append(french)
                if ref:
                    current["ref"].append(ref)
        flush()
    candidates = deduplicate(pd.DataFrame(records))
    accepted = candidates[candidates["review_status"] != "quarantine"].reset_index(drop=True)
    quarantined = candidates[candidates["review_status"] == "quarantine"].reset_index(drop=True)
    return accepted, quarantined

pdf_bitext, pdf_quarantine = extract_pdf_rows(PDF_PATH)
print(f"PDF accepted candidates: {len(pdf_bitext)}")
print(f"PDF quarantined candidates: {len(pdf_quarantine)}")
display(pdf_bitext.head(20))

In [ ]:
# Quick PDF quality checks: Hebrew/layout artifacts must not reach accepted rows.
assert not pdf_bitext["mashi"].str.contains(HEBREW_RE).any()
assert not pdf_bitext["french"].str.contains(HEBREW_RE).any()
assert not pdf_bitext["french"].str.contains(r"(?<!\w)Dt(?!\w)", regex=True).any()
assert pdf_bitext["ref"].str.fullmatch(REF_RE).all()
if not pdf_quarantine.empty:
    display(pdf_quarantine[["ref", "mashi", "french", "quality_flags"]].head(20))
print(pdf_bitext.sample(min(10, len(pdf_bitext)), random_state=42)[["ref", "mashi", "french"]].to_string(index=False))

## 5. Combine, inspect, and save

The files use real JSON Lines format—one JSON object per line—and UTF-8 CSV. Do not use the combined file for evaluation before manually reviewing `needs_review` rows and separating train/dev/test data by source.

In [ ]:
frames = [df for df in [bible_bitext, murhula_bitext, pdf_bitext] if not df.empty]
all_bitext = deduplicate(pd.concat(frames, ignore_index=True))

all_bitext.to_csv("mashi-french_bitext_candidates.csv", index=False, encoding="utf-8")
all_bitext.to_json(
    "mashi-french_bitext_candidates.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
bible_bitext.to_json("mashi-french_ebible_exodus.jsonl", orient="records", lines=True, force_ascii=False)
murhula_bitext.to_json("mashi-french_murhula.jsonl", orient="records", lines=True, force_ascii=False)
pdf_bitext.to_json("mashi-french_deuteronomy_dictionary.jsonl", orient="records", lines=True, force_ascii=False)
pdf_quarantine.to_json("mashi-french_deuteronomy_quarantine.jsonl", orient="records", lines=True, force_ascii=False)

print(all_bitext.groupby(["source", "unit_type"]).size())
print(f"\nTotal unique candidate pairs: {len(all_bitext)}")
display(all_bitext.sample(min(20, len(all_bitext)), random_state=42))

In [ ]:
# Optional Colab downloads
try:
    from google.colab import files
    for filename in [
        "mashi-french_bitext_candidates.csv",
        "mashi-french_bitext_candidates.jsonl",
        "mashi-french_ebible_exodus.jsonl",
        "mashi-french_murhula.jsonl",
        "mashi-french_deuteronomy_dictionary.jsonl",
        "mashi-french_deuteronomy_quarantine.jsonl",
    ]:
        files.download(filename)
except ImportError:
    print("Not running in Colab; output files remain in the working directory.")

## Required manual review

Before model training or BLEU/chrF evaluation:

1. inspect every murhula.com and PDF row;
2. remove headings, generated analyses, partial phrases, and misaligned wrapped cells;
3. verify Mashi spelling/diacritics with a speaker;
4. preserve source, URL, verse/reference, edition, and licence metadata;
5. keep data from the same Bible edition/source on only one side of a train/dev/test split to avoid leakage;
6. obtain permission before bulk reuse of murhula.com or the PDF—the notebook's limited public-page access does not create a new licence.